In [1]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, AutoModelForQuestionAnswering, BitsAndBytesConfig
import torch

/home/mrosaria/Projects/NLP/GymRat/ratenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Get Chunks

In [2]:
import json
from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain.document_loaders import PyMuPDFLoader, PDFMinerLoader
def document_loader(file):
    #PyMuPDFLoader best for large books
    loader = PyMuPDFLoader(file)
    loaded_document = loader.load()
    return loaded_document

def split_text(document, chunk_size=500, chunk_overlap=50, return_as_documents=True):    
    separators = ["\n\n", "\n", ".", " ", ""]
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len, 
        separators=separators
    )
    if return_as_documents:
        return text_splitter.split_documents(document)
    else:
        return text_splitter.split_text(document)


file="../data/books/Rebuilding Milo The Lifters Guide to Fixing Common Injuries and Building a Strong Foundation for Enhancing Performance (Dr. Aaron Horschig, Kevin Sonthana) (Z-Library).pdf"
document = document_loader(file)
document_clean = []
for page in document:
    if(len(page.page_content) >= 50):
        document_clean.append(page)

chunks = []
for file in [file]:
    #album_path = os.path.join(all_albums_path, album_path)
    chunk = split_text(document_clean, 500, 50)
    chunks.extend(chunk)
len(chunks)

1726

In [3]:
def contains_private_unicode(text):
    return any('\uE000' <= char <= '\uF8FF' for char in text)

def find_words_with_bad_glyphs(text):
    words = text.split()
    return [word for word in words if contains_private_unicode(word)]

def find_non_english_characters(text):
    return set(char for char in text if ord(char) > 127)

def find_words_with_non_english_chars(text):
    words = text.split()
    return [word for word in words if any(ord(c) > 127 for c in word)]



In [13]:
# words_with_non_english_chars, words_with_bad_glyphs,non_english_characters = [], [], []
# for chunk in chunks:
#     text = chunk.page_content
#     words_with_non_english_chars.append(find_words_with_non_english_chars(text))

#     words_with_bad_glyphs.append(find_words_with_bad_glyphs(text))

#     non_english_characters.append(find_non_english_characters(text))

words_with_non_english_chars, words_with_bad_glyphs, non_english_characters = [], [], []
for chunk in chunks:
    text = chunk.page_content
    words_with_non_english_chars.extend(find_words_with_non_english_chars(text))

    words_with_bad_glyphs.extend(find_words_with_bad_glyphs(text))

    non_english_characters.extend(find_non_english_characters(text))

In [72]:
unique_words_with_non_english_chars = list(set(words_with_non_english_chars))
unique_words_with_bad_glyphs = list(set(words_with_bad_glyphs))
unique_non_english_characters = list(set(non_english_characters))

In [288]:
unique_non_english_characters[20:]

['ã', 'ø', '•', '»', 'ä', 'é', 'ﬂ', 'ü', 'ë', 'ͅ', 'ł', 'ﬃ', '®', 'ï', 'Š']

In [289]:
def replace_smart_quotes(text):
    text = text.replace("–", "-").replace("»", "")
    return text.replace('’', "'").replace('‘', "'").replace('“', '"').replace('”', '"').replace("—", "-").replace("/", " ").replace("®", "").replace("…", "...").replace("×", "x")

In [ ]:
import unicodedata
#NFKC helps convert compatible characters into standard ones, e.g., ’ → ', ﬁ → fi. (diﬀers word)
def normalize_quotes(text):
    return unicodedata.normalize("NFKC", text)


In [ ]:
replacements = {
    "e\u0346": "fl"}

In [240]:
def replace_bad_words(word):
    letters_combo = ["e\u0346"]
    word = unique_words_with_non_english_chars[4]
    for n in range(len(word)):
        letter = word[n]
        if letter in unique_non_english_characters:
            letter_to_be_replaced=""
            if "e"+letter in letters_combo:
                #print("e"+letter, replacements["e"+letter])
                letter_to_be_replaced = "e"+letter
                value = letter_to_be_replaced
            
            if letter in replacements.keys():
                value = value if letter_to_be_replaced else letter
            word = word.replace(letter, replacements[value])
            return word
                #print(letter, word, replacements[letter] ,f'--->  {"\u0346"}  ' ,word.replace(letter, replacements[value] ))

word = unique_words_with_non_english_chars[4]
new_word = replace_bad_words(word)
new_word

'flexed/rounded'

In [248]:
type(unique_words_with_non_english_chars[0])

str

In [ ]:
def remove_number_range_with_unicode_dash(text):
    # Match patterns like 211–20 or 211–220. with an optional period
    return re.sub(r'\b\d+[\u2013\u2014\u2012\u2010\u2212-]\d+\.*', '', text)

#replace_smart_quotes, normalize_quotes


In [295]:
bad_words = unique_words_with_non_english_chars
len(bad_words)

1834

In [292]:
len([remove_number_range_with_unicode_dash(i) for i in bad_words if remove_number_range_with_unicode_dash(i)])

1602

In [296]:
bad_words = [remove_number_range_with_unicode_dash(i) for i in bad_words]

In [ ]:
bad_words = [replace_smart_quotes(i) for i in bad_words]
bad_words = [normalize_quotes(i) for i in bad_words]




In [299]:
list(set([i for i in bad_words if i]))

["I'm",
 'sufficient.',
 'off',
 '"link"',
 "They're",
 '"Diagnostics',
 '"Are',
 'slightly-an',
 '"Quick',
 'ͅnger',
 'cases,"',
 'hypertrophy,"',
 '"Looped',
 '"lock',
 'cohort,"',
 'oxygen,"',
 '"Australian',
 '͆exors,',
 'ͅxing',
 'efficacy',
 'in͆exible',
 '"Forces',
 'offices',
 'on."',
 '"Adaptational',
 '"stiff"',
 'fluid',
 '͆ip',
 '͆exing',
 'Inflammation:',
 'effectiveness!',
 'effective,',
 '"Epidemiological',
 'sports"',
 'bursitis,"',
 '͆ow)',
 'armpits"',
 'first',
 '"I',
 '"twist"',
 'chain"',
 'movement-no',
 '"double-jointed,"',
 'stiffening',
 '"farmer"',
 '"winds',
 'efficient',
 '©Bruce',
 'down."',
 '"Overuse',
 'approach,"',
 "That's",
 'deͅned',
 'lift-from',
 '"healthy"',
 '"Studies',
 '͆at',
 'stuff',
 '"reactive',
 "aren't",
 '"tuck',
 'ͅnish',
 'prevention,"',
 '"reactive,"',
 'length,"',
 'options,"',
 'Maffiuletti,',
 '"spine-friendly"',
 'tension?"',
 '"defense',
 '"Differences',
 '"Biomechanics',
 'multiͅdus',
 'in͆ammation,',
 '"blocked"',
 '4-',
 'quic

In [264]:
for word in bad_words:
    print(word)
    #print(replace_smart_quotes(word))



injuries—and
“Core
(jumper’s
͆exed/rounded
“Physiological
diﬀers.
;
Eﬀects
“more
“C.”
mobility/͆exibility
chemokines,”
We’re
ͅnger
—a
Reinl’s
“progressive

D’Haen,
stiﬀ/tight
;
“pop”
“tripod
swelling”
cuﬀ),
maneuvers,”
Here’s
‘on
weightlifters,”

͆exors,
ͅxing
it!”
presentations,”
in͆exible

‘tensioners’

“Reverse

damage,”
stiﬀ.

;

“reactive-on-degeneration”
football—analysis
“Eﬃcacy
“neutral”
͆ip
wife’s


͆exing

recommendations,”
͆ow)
can”
“Lateral
suﬀering.
tendinosis,”
“Arthrogenic
That’s
dysplasia,”
advantage.”

“Humeral
;


remodeling”

diﬀerence—maybe
Woodruﬀ,

©Bruce
deͅcit.”
“loose”

“V”:
deͅned
eﬀusion
eﬀort?
“crunch”
intolerant”
“Reliability
͆at
“winds
“SLAP
syndrome—long
ͅnish
lifter’s
;
“link”
“Neuromuscular
diﬀerent.
“1,000
;
;
multiͅdus
in͆ammation,
“trauma”
“excessive
what?—evacuate
stiﬀ,
squat,”
“One-footed
“textbook”
ͅrst,
“row”
“Diagnosis

head.”
gender,”

that’s


;
re͆exively
“reactive
pockets”
program,”
“crossover
seasons,”

“Faster
“It
pain,”
ACSM’s
eﬀortless


In [ ]:
[replace_smart_quotes(i) for i in ]

TypeError: 'str' object cannot be interpreted as an integer

In [92]:
[i for i in unique_words_with_non_english_chars[4]]

['͆', 'e', 'x', 'e', 'd', '/', 'r', 'o', 'u', 'n', 'd', 'e', 'd']

In [99]:
unique_words_with_non_english_chars[4]

'͆exed/rounded'

In [229]:
replacements = {
    "\u0346": "e", 
    "e\u0346": "fl"
    }
replacements

{'͆': 'e', 'e͆': 'fl'}

In [ ]:
letters_combo = ["e\u0346"]
word = unique_words_with_non_english_chars[4]
for n in range(len(word)):
    letter = word[n]
    if letter in unique_non_english_characters:
        #print(unique_words_with_non_english_chars[4],' -> letter:', letter,'---', letter+'e')
        letter_to_be_replaced=""
        if "e"+letter in letters_combo:
            #print("e"+letter, replacements["e"+letter])
            letter_to_be_replaced = "e"+letter
            value = letter_to_be_replaced
        
        if letter in replacements.keys():
            value = value if letter_to_be_replaced else letter
            print(letter, word, replacements[letter] ,f'--->  {"\u0346"}  ' ,word.replace(letter, replacements[value] ))
            

͆ ͆exed/rounded e --->  ͆   flexed/rounded


In [ ]:
for letter in unique_words_with_non_english_chars[4]:
    if letter in unique_non_english_characters:
        print(unique_words_with_non_english_chars[4],' -> letter:', letter,'---', letter+'e')
        if 

        
        for smart, ascii_equiv in replacements.items():
            if letter
            print(f"smart: {smart} , ascii_equiv {ascii_equiv}, letter: {letter} , e+letter: {"e"+letter}")
            letters_combo = "e"+letter
            if replacements.keys(letters_combo)
                print(unique_words_with_non_english_chars[4].replace(smart, ascii_equiv ))
#if unique_words_with_non_english_chars[4].split('/')[0][0] in unique_non_english_characters:


͆exed/rounded  -> letter: ͆ --- ͆e
smart: ͆ , ascii_equiv e, letter: ͆ , e+letter: e͆
eexed/rounded
smart: e͆ , ascii_equiv fl, letter: ͆ , e+letter: e͆
͆exed/rounded


In [ ]:
import unicodedata

def strip_accents(text):
    return ''.join(
        c for c in unicodedata.normalize('NFD', text)
        if not unicodedata.combining(c)
    )

print(strip_accents(unique_words_with_non_english_chars[4]))

exed/rounded


In [133]:
{"e\u0346": "fl"}

{'e͆': 'fl'}

In [57]:
normalize_quotes(unique_words_with_non_english_chars[4])

'͆exed/rounded'

In [70]:
replacements = {
    "doesn’t": "doesn't",
    "e͆xion": "flexion",
    "“quotes”": '"quotes"'
}
import re
def replace_with_mapping(text, replacements: dict):
    words = re.findall(r'\S+', text)
    new_words = []
    for word in words:
        norm = unicodedata.normalize('NFD', word)
        if any(ord(c) > 127 for c in norm) and word in replacements:
            new_words.append(replacements[word])
        else:
            new_words.append(word)
    return ' '.join(new_words)


In [71]:
print(replace_with_mapping(unique_words_with_non_english_chars[4].split('/')[0], replacements))


͆exed


In [78]:
if unique_words_with_non_english_chars[4].split('/')[0][0] in unique_non_english_characters:
    print(unique_words_with_non_english_chars[4].split('/')[0])

͆exed


In [50]:
unique_words_with_non_english_chars

['211–20.',
 'injuries—and',
 '“Core',
 '(jumper’s',
 '͆exed/rounded',
 '“Physiological',
 'diﬀers.',
 '672–81;',
 'Eﬀects',
 '“more',
 '“C.”',
 'mobility/͆exibility',
 'chemokines,”',
 'We’re',
 'ͅnger',
 '—a',
 'Reinl’s',
 '“progressive',
 '36–41.',
 'D’Haen,',
 'stiﬀ/tight',
 '385–94;',
 '“pop”',
 '“tripod',
 'swelling”',
 'cuﬀ),',
 'maneuvers,”',
 'Here’s',
 '‘on',
 'weightlifters,”',
 '72–9.',
 '͆exors,',
 'ͅxing',
 'it!”',
 'presentations,”',
 'in͆exible',
 '173–8.',
 '‘tensioners’',
 '106–19.',
 '“Reverse',
 '71–81.',
 'damage,”',
 'stiﬀ.',
 '388–95.',
 '169–81;',
 '827–38.',
 '“reactive-on-degeneration”',
 'football—analysis',
 '“Eﬃcacy',
 '“neutral”',
 '͆ip',
 'wife’s',
 '63–70.',
 '805–10.',
 '͆exing',
 '128–36.',
 'recommendations,”',
 '͆ow)',
 'can”',
 '“Lateral',
 'suﬀering.',
 'tendinosis,”',
 '“Arthrogenic',
 'That’s',
 'dysplasia,”',
 'advantage.”',
 '3084–7.',
 '“Humeral',
 '358–69;',
 '44–8.',
 '336–9.',
 'remodeling”',
 '375–9.',
 'diﬀerence—maybe',
 'Woodruﬀ,',
 '95

In [89]:
unique_on_english_characters

['ê',
 '’',
 'Ł',
 'ń',
 '—',
 '”',
 '–',
 'ﬀ',
 '͆',
 '‘',
 '½',
 'ö',
 '“',
 'Ö',
 '×',
 '…',
 'Ø',
 '©',
 'ﬁ',
 'ğ',
 'ã',
 'ø',
 '•',
 '»',
 'ä',
 'é',
 'ﬂ',
 'ü',
 'ë',
 'ͅ',
 'ł',
 'ﬃ',
 '®',
 'ï',
 'Š']

In [35]:
if unique_on_english_characters[0] in [i for i in unique_words_with_non_english_chars[4]]):
    print(unique_words_with_non_english_chars[4])

SyntaxError: unmatched ')' (20458738.py, line 1)

In [28]:
unique_on_english_characters

['ê',
 '’',
 'Ł',
 'ń',
 '—',
 '”',
 '–',
 'ﬀ',
 '͆',
 '‘',
 '½',
 'ö',
 '“',
 'Ö',
 '×',
 '…',
 'Ø',
 '©',
 'ﬁ',
 'ğ',
 'ã',
 'ø',
 '•',
 '»',
 'ä',
 'é',
 'ﬂ',
 'ü',
 'ë',
 'ͅ',
 'ł',
 'ﬃ',
 '®',
 'ï',
 'Š']

In [8]:
words_with_non_english_chars, words_with_bad_glyphs,non_english_characters = [], [], []
for chunk in chunks:
    text = chunk.page_content
    words_with_non_english_chars.extend(find_words_with_non_english_chars(text))

In [10]:
list(set(words_with_non_english_chars))

['211–20.',
 'injuries—and',
 '“Core',
 '(jumper’s',
 '͆exed/rounded',
 '“Physiological',
 'diﬀers.',
 '672–81;',
 'Eﬀects',
 '“more',
 '“C.”',
 'mobility/͆exibility',
 'chemokines,”',
 'We’re',
 'ͅnger',
 '—a',
 'Reinl’s',
 '“progressive',
 '36–41.',
 'D’Haen,',
 'stiﬀ/tight',
 '385–94;',
 '“pop”',
 '“tripod',
 'swelling”',
 'cuﬀ),',
 'maneuvers,”',
 'Here’s',
 '‘on',
 'weightlifters,”',
 '72–9.',
 '͆exors,',
 'ͅxing',
 'it!”',
 'presentations,”',
 'in͆exible',
 '173–8.',
 '‘tensioners’',
 '106–19.',
 '“Reverse',
 '71–81.',
 'damage,”',
 'stiﬀ.',
 '388–95.',
 '169–81;',
 '827–38.',
 '“reactive-on-degeneration”',
 'football—analysis',
 '“Eﬃcacy',
 '“neutral”',
 '͆ip',
 'wife’s',
 '63–70.',
 '805–10.',
 '͆exing',
 '128–36.',
 'recommendations,”',
 '͆ow)',
 'can”',
 '“Lateral',
 'suﬀering.',
 'tendinosis,”',
 '“Arthrogenic',
 'That’s',
 'dysplasia,”',
 'advantage.”',
 '3084–7.',
 '“Humeral',
 '358–69;',
 '44–8.',
 '336–9.',
 'remodeling”',
 '375–9.',
 'diﬀerence—maybe',
 'Woodruﬀ,',
 '95

In [6]:
unique_words_with_non_english_chars = []
for words in words_with_non_english_chars:
    if not words:
        continue
    else:
        unique_words_with_non_english_chars.extend(i)

unique_words_with_bad_glyphs = []
for words in words_with_bad_glyphs:
    if not words:
        continue
    else:
        unique_words_with_bad_glyphs.extend(i)

unique_non_english_characters = []
for words in non_english_characters:
    if not words:
        continue
    else:
        unique_non_english_characters.extend(i)

unique_words_with_non_english_chars = list(set(unique_words_with_non_english_chars))
unique_words_with_bad_glyphs = list(set(unique_words_with_bad_glyphs))
unique_non_english_characters = list(set(unique_non_english_characters))

NameError: name 'i' is not defined

In [7]:
unique_words_with_non_english_chars, unique_words_with_bad_glyphs, unique_non_english_characters

NameError: name 'unique_words_with_bad_glyphs' is not defined

In [120]:
unique_non_english_characters

['Don’t',
 'Oﬀ!',
 'In͆ammation',
 'Don’t',
 'Oﬀ!',
 'In͆ammation',
 'Don’t',
 'Oﬀ!',
 'In͆ammation',
 'Don’t',
 'Oﬀ!',
 'In͆ammation',
 'Don’t',
 'Oﬀ!',
 'In͆ammation',
 'Don’t',
 'Oﬀ!',
 'In͆ammation',
 'Don’t',
 'Oﬀ!',
 'In͆ammation',
 'Don’t',
 'Oﬀ!',
 'In͆ammation',
 'Don’t',
 'Oﬀ!',
 'In͆ammation',
 'Don’t',
 'Oﬀ!',
 'In͆ammation',
 'Don’t',
 'Oﬀ!',
 'In͆ammation',
 'Don’t',
 'Oﬀ!',
 'In͆ammation',
 'Don’t',
 'Oﬀ!',
 'In͆ammation',
 'Don’t',
 'Oﬀ!',
 'In͆ammation',
 'Don’t',
 'Oﬀ!',
 'In͆ammation',
 'Don’t',
 'Oﬀ!',
 'In͆ammation',
 'Don’t',
 'Oﬀ!',
 'In͆ammation',
 'Don’t',
 'Oﬀ!',
 'In͆ammation',
 'Don’t',
 'Oﬀ!',
 'In͆ammation',
 'Don’t',
 'Oﬀ!',
 'In͆ammation',
 'Don’t',
 'Oﬀ!',
 'In͆ammation',
 'Don’t',
 'Oﬀ!',
 'In͆ammation',
 'Don’t',
 'Oﬀ!',
 'In͆ammation',
 'Don’t',
 'Oﬀ!',
 'In͆ammation',
 'Don’t',
 'Oﬀ!',
 'In͆ammation',
 'Don’t',
 'Oﬀ!',
 'In͆ammation',
 'Don’t',
 'Oﬀ!',
 'In͆ammation',
 'Don’t',
 'Oﬀ!',
 'In͆ammation',
 'Don’t',
 'Oﬀ!',
 'In͆ammation',
 'Don’t',
 'Oﬀ

In [81]:
text = chunks[110].page_content

In [80]:
find_words_with_non_english_chars(chunks[110].page_content)

['it’s', '͆exion,', 'advantage.”']

In [84]:
text.replace('͆exion,', 'flexion')

'without creating a stress concentration at a single level but create a gentle\ncurve (and some of the great powerlifters have quite a bit more in the\nthoracic spine) to allow the mechanics to pull around the knee. So it’s a\nlittle bit of flexion and then they lock it in place. When you measure this,\nthe motion is still around the hips. This is the second best in terms of\nspinal stress and resilience to injury. For some lifters, it creates a\nmechanical advantage.” 21'

In [ ]:
text[237]

'͆'

In [73]:
print(chunks[110].page_content)

without creating a stress concentration at a single level but create a gentle
curve (and some of the great powerlifters have quite a bit more in the
thoracic spine) to allow the mechanics to pull around the knee. So it’s a
little bit of ͆exion, and then they lock it in place. When you measure this,
the motion is still around the hips. This is the second best in terms of
spinal stress and resilience to injury. For some lifters, it creates a
mechanical advantage.” 21


In [ ]:
from langchain.document_loaders import PyMuPDFLoader, PDFMinerLoader, PDFPlumberLoader

def document_loader(file):
    #PyMuPDFLoader best for large books
    loader = PDFPlumberLoader(file)
    loaded_document = loader.load()
    return loaded_document

ImportError: cannot import name 'PDFMinerPDFasPagesLoader' from 'langchain.document_loaders' (/home/mrosaria/Projects/NLP/GymRat/ratenv/lib/python3.12/site-packages/langchain/document_loaders/__init__.py)

In [63]:
loader = PDFMinerLoader(file)
doc = loader.load()

Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBB

In [64]:
doc

[Document(metadata={'producer': 'calibre (5.9.0) [https://calibre-ebook.com]', 'creator': 'Soda PDF', 'creationdate': '2021-01-19T08:42:12+00:00', 'author': 'Aaron Horschig & Kevin Sonthana', 'moddate': '2021-01-19T12:21:14+00:00', 'title': "Rebuilding Milo: The Lifter's Guide to Fixing Common Injuries and Building a Strong Foundation for Enhancing Performance", 'total_pages': 585, 'source': '../data/books/Rebuilding Milo The Lifters Guide to Fixing Common Injuries and Building a Strong Foundation for Enhancing Performance (Dr. Aaron Horschig, Kevin Sonthana) (Z-Library).pdf'}, page_content='\n\x0c\n\x0cFirst published in 2021 by Victory Belt Publishing Inc.\n\nCopyright © 2021 Aaron Horschig and Dr. Kevin Sonthana\n\nAll rights reserved\nNo part of this publication may be reproduced or distributed\nin any form or by any means, electronic or mechanical, or\nstored in a database or retrieval system, without prior written\npermission from the publisher.\n\nISBN-13: 99 114 111 107 101 114

In [58]:
doc = document_loader(file)

Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBB

In [62]:
print(doc[40].page_content)

some elite powerlifters have been known to hunch or round their backs
purposefully. 20
But doesn’t this go against everything we just learned? Yes and no. To
start, elite powerlifters who lift with this technique aren’t usually allowing
their back to move into more  exion as the lift is pulled. They’re instead
bracing or “locking in” their spine with a slight curvature in the spine,
maintaining that degree of  exion and moving about the hips to
complete the deadlift. The late Konstantin Konstantinov was a great
example of this; you can  nd many of his legendary lifts on YouTube.
During one of my conversations with Dr. McGill on this exact topic, he
had the following to say: “In terms of spinal stress and resilience, locking
the spine in a neutral posture and moving entirely about the fulcrum of
the hip would be the most resilient for the spine. However, you can  ex
without creating a stress concentration at a single level but create a gentle
curve (and some of the great powerlifters ha

In [51]:
print(document_loader(file)[40].page_content)

Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBB

IndexError: list index out of range

# Documentation Loading a Model from hugging face

🔍 Code Explanation

```python
tokenizer = AutoTokenizer.from_pretrained(model_name)
```

### 👉 What this does:

* **Loads the tokenizer** associated with the model `model_name` from Hugging Face.
* The tokenizer is responsible for **converting text into tokens** (numerical representations) and back from tokens to text.

### 🔧 `AutoTokenizer`:

* A Hugging Face **auto-class** that automatically selects the right tokenizer class based on the model.
* Example: if `model_name` is `"gpt2"`, it will internally use `GPT2Tokenizer`.

---

```python
model = AutoModelForCausalLM.from_pretrained(model_name)
```

### 👉 What this does:

* Loads the **pretrained language model** corresponding to `model_name`, specifically for **Causal Language Modeling** (CLM).
* **CLM** means predicting the **next word in a sequence**, which is the task used in models like GPT.

### 🔧 `AutoModelForCausalLM`:

* This is a class for **text generation models**, like:

  * GPT-2 / GPT-Neo / LLaMA / Mistral / Falcon / Phi
* It automatically picks the right architecture for your model.

### 🧠 Other Options:

Depending on your task, you might use:

| AutoModel Type                       | Use Case                                                             |
| ------------------------------------ | -------------------------------------------------------------------- |
| `AutoModel`                          | Generic model without heads (usually for embeddings or custom heads) |
| `AutoModelForSequenceClassification` | Text classification                                                  |
| `AutoModelForQuestionAnswering`      | Extractive QA                                                        |
| `AutoModelForTokenClassification`    | NER / POS tagging                                                    |
| `AutoModelForSeq2SeqLM`              | Translation / summarization                                          |
| `AutoModelForCausalLM`               | Text generation (e.g., GPT-style)                                    |
| `AutoModelForMaskedLM`               | BERT-style masked word prediction                                    |

---

```python
qa_gen = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=200)
```

### 👉 What this does:

* Creates a Hugging Face **`pipeline`** for text generation.
* Wraps the tokenizer and model into an **easy-to-use interface** for generation tasks.
* `max_new_tokens=200` means the model can generate **up to 200 new tokens** after the input.

---

## 🛠️ What is `pipeline`?

`pipeline()` is Hugging Face’s high-level API to **simplify the use of models** for common NLP tasks. It takes care of:

* Preprocessing (tokenization)
* Running the model
* Postprocessing (decoding)

---

## 🧪 Supported Tasks in `pipeline()` and Their Uses

| Task                            | Description                                                | Model Type               |
| ------------------------------- | ---------------------------------------------------------- | ------------------------ |
| `"text-generation"`             | Generate continuation of a prompt                          | GPT-2, LLaMA, Mistral    |
| `"text2text-generation"`        | Converts input to output text (e.g., summarize, translate) | T5, BART                 |
| `"summarization"`               | Summarizes long text                                       | T5, BART                 |
| `"translation"`                 | Translates between languages                               | MarianMT, T5             |
| `"question-answering"`          | Extracts answer from a given context                       | BERT, RoBERTa            |
| `"zero-shot-classification"`    | Classifies text into labels without training               | BART, RoBERTa            |
| `"sentiment-analysis"`          | Predicts sentiment (positive/negative)                     | DistilBERT, RoBERTa      |
| `"ner"`                         | Named Entity Recognition                                   | BERT, RoBERTa            |
| `"fill-mask"`                   | Predicts missing words (BERT-style)                        | BERT                     |
| `"text-classification"`         | General classification (spam/ham, topic, etc.)             | Any classification model |
| `"conversational"`              | Used for dialogue agents                                   | DialoGPT                 |
| `"image-classification"`        | For images (ViT, ResNet)                                   | Vision models            |
| `"document-question-answering"` | QA on scanned documents                                    | LayoutLM                 |
| `"table-question-answering"`    | QA on tables                                               | TAPAS                    |
| `"speech-to-text"`              | Transcribes audio to text                                  | Whisper, Wav2Vec2        |
| `"text-to-speech"`              | Text-to-voice                                              | Bark, TTS                |
| `"feature-extraction"`          | Get vector embeddings                                      | Any transformer          |

---

## ✅ Summary

| Line                                   | What it does                                               |
| -------------------------------------- | ---------------------------------------------------------- |
| `AutoTokenizer.from_pretrained`        | Loads a tokenizer that can convert text to tokens and back |
| `AutoModelForCausalLM.from_pretrained` | Loads a pretrained model for text generation               |
| `pipeline("text-generation", ...)`     | Creates an easy interface for generating text              |

You can now **generate**, **fine-tune**, or **serve** these models in apps like Gradio!

Would you like an example of wrapping this into a chatbot-style Gradio interface next?


# Code Mistral-7b

In [ ]:
# model_id = "mistralai/Mistral-7B-v0.1"

# tokenizer = AutoTokenizer.from_pretrained(model_id)

# # Define quantization config
# quant_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4",  # You can also use "fp4"
#     bnb_4bit_compute_dtype="float16"
# )

# model = AutoModelForCausalLM.from_pretrained(
#     model_id,
#     device_map="auto",
#     quantization_config=quant_config,
#     trust_remote_code=True
# )

# generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

Fetching 2 files:   0%|          | 0/2 [01:00<?, ?it/s]


# Code Llama

In [4]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [5]:
#Loads the tokenizer associated with the model model_name from Hugging Face
#AutoTokenizer A Hugging Face auto-class that automatically selects the right tokenizer class based on the model.
#Example: if model_name is "gpt2", it will internally use GPT2Tokenizer.
#tokenizer = AutoTokenizer.from_pretrained(model_name)
#Pad the token 
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
#eos -> end of string token is the pad token
tokenizer.pad_token = tokenizer.eos_token


#The line code loads the pretrained language model corresponding to model_name, specifically for Causal Language Modeling (CLM).
#CLM means predicting the next word in a sequence, which is the task used in models like GPT.
#AutoModelForCausalLM:  class for text generation models, like:GPT-2 / GPT-Neo / LLaMA / Mistral / Falcon / Phi
#It automatically picks the right architecture for your model.

model = AutoModelForCausalLM.from_pretrained(model_name)

#Hugging Face pipeline for text generation.
#Wraps the tokenizer and model into an easy-to-use interface for generation tasks
#pipeline is Hugging Face’s high-level API to simplify the use of models for common NLP tasks. It takes care of: Preprocessing (tokenization), Running the modeland Postprocessing (decoding)
qa_gen = pipeline("text-generation", model=model, tokenizer=tokenizer,  max_new_tokens=256)


Device set to use cuda:0


In [ ]:
def get_prompt(text):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful and concise medical tutor. "
                "Based on the text below, generate a JSON object with a single question-answer pair "
                "using the keys 'instruction' for the question and 'output' for the answer.\n"
                "Respond only with the JSON object."
            ),
        },
        {"role": "user", "content": text},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Function to generate QA from text

prompt = get_prompt(chunks[103].page_content)

output = qa_gen(prompt, max_new_tokens=256, do_sample=True, temperature=0.7, top_k=50, top_p=0.95)[0]["generated_text"]


In [34]:
raw_output = output.split("<|assistant|>")[1]


In [ ]:
# Step 1: Remove '\n{ and add missing clossing braket
clean_json_str = raw_output.strip() if raw_output.strip()[-1] == "}" else raw_output.strip()+"}"
# Step 2: Parse JSON
data = json.loads(clean_json_str)

In [38]:
data

{'question': 'What is the mechanism behind the development of a disc bulge, and how does it affect injury risk?',
 'answer': 'The mechanism behind the development of a disc bulge is ͆exion, which is the process of ͆exing the spine as it ͆exes. If the force applied to the spine as it ͆exes is low, power generation remains low, and so does injury risk. This is why the cat-camel model, which simulates low-force disc bulging, is a useful tool for understanding the mechanics of disc injury risk.'}

In [41]:
print(chunks[103].page_content)

anatomy, genetics, the amount of weight lifted, and the degree of poor
technique, your body may be more or less resilient to developing a disc
bulge. 18
This doesn’t mean you should fear ͆exion of the spine. However, you
must understand that the mechanism that creates a disc bulge includes
͆exion. If the force applied to the spine as it ͆exes is low, power
generation remains low, and so does injury risk. This is why the cat-camel


In [37]:
# Step 3: Save to file
with open('output.json', 'w') as f:
    json.dump(data, f, indent=2)

In [ ]:
# Step 1: Remove triple backticks and 'json' label
clean_json_str = raw_output.strip().strip('`json').strip('```')

# Step 2: Parse JSON
data = json.loads(clean_json_str)



In [20]:
def make_prompt(chunk):
    return f"""### Instruction:
You are a medical tutor. Read the following text and generate 1-2 question with their answer pairs that could be used to train a chatbot.

### Context:
\"\"\"{chunk}\"\"\"

### Output format (JSON list):
[
  {{
    "instruction": "",
    "input": "",
    "output": "Explanation ..."
  }}
]
"""
text = chunk[100].page_content

In [25]:
prompt = make_prompt(text)
output = qa_gen(prompt, max_new_tokens=512, do_sample=True, temperature=0.7)[0]['generated_text']
output.split("(JSON list)")[1]


':\n[\n  {\n    "instruction": "",\n    "input": "",\n    "output": "Explanation ..."\n  }\n]\n\n### Question 1:\n- Given the text: "Now, if you are just performing a few bodyweight squats and butt\nwinking occurs, it’s likely not a big deal. Minimal power is generated at the\nspine during a normal-tempo air squat. However, as soon as you add a\nbarbell, things change. If butt winking continues under load, the power\ngenerated at the spine increases at one or two speciͅc joints of the\nlumbar spine (usually L4/5 and L5/S1). Therefore, when you have a stress\nconcentration of power at one or two lumbar segments, injury risk"\n\n- Explanation: If butt winking continues under load, the power generated at the spine increases at one or two speciͅc joints of the lumbar spine. This increases risk of injury.\n\n### Question 2:\n- Given the text: "Minimal power is generated at the spine during a normal-tempo air\nsquat. However, as soon as you add a barbell, things change. If butt\nwinking cont

In [24]:
output.split("(JSON list)")[1]


':\n[\n  {\n    "instruction": "",\n    "input": "",\n    "output": "Explanation ..."\n  }\n]\n\n### Question 1:\nQuestion: How does a stress concentration at the spine increase the risk of injury during a strength training session? Answer: A stressor may result in a concentration of power at one or two lumbar segments, leading to an increased risk of injury during a strength training session.'

In [28]:
text

'Now, if you are just performing a few bodyweight squats and butt\nwinking occurs, it’s likely not a big deal. Minimal power is generated at the\nspine during a normal-tempo air squat. However, as soon as you add a\nbarbell, things change. If butt winking continues under load, the power\ngenerated at the spine increases at one or two speciͅc joints of the\nlumbar spine (usually L4/5 and L5/S1). Therefore, when you have a stress\nconcentration of power at one or two lumbar segments, injury risk'

In [ ]:
############# HEREEEEEE

In [ ]:
def get_prompt(text):
    messages = [
    {
        "role": "system",
        "content": "You are a medical tutor. Read the following text and generate one question with its answer that could be used to train a chatbot.",
    },
    {"role": "user", "content": text},
]
    return messages
text = chunks[103].page_content
prompt = qa_gen.tokenizer.apply_chat_template(get_prompt(text), tokenize=False, add_generation_prompt=True)

outputs = qa_gen(prompt, max_new_tokens=256, do_sample=True, temperature=0.7, top_k=50, top_p=0.95)
print(outputs[0]["generated_text"])


In [ ]:
def get_prompt(text):
    messages = [
    {
        "role": "system",
        "content": "You are a medical tutor. Read the following text and generate one question with its answer that could be used to train a chatbot.",
    },
    {"role": "user", "content": text},
]
    return messages


In [56]:
text = chunks[103].page_content
prompt = qa_gen.tokenizer.apply_chat_template(get_prompt(text), tokenize=False, add_generation_prompt=True)

outputs = qa_gen(prompt, max_new_tokens=256, do_sample=True, temperature=0.7, top_k=50, top_p=0.95)
print(outputs[0]["generated_text"])


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


<|system|>
You are a medical tutor. Read the following text and generate one question with its answer that could be used to train a chatbot.</s>
<|user|>
anatomy, genetics, the amount of weight lifted, and the degree of poor
technique, your body may be more or less resilient to developing a disc
bulge. 18
This doesn’t mean you should fear ͆exion of the spine. However, you
must understand that the mechanism that creates a disc bulge includes
͆exion. If the force applied to the spine as it ͆exes is low, power
generation remains low, and so does injury risk. This is why the cat-camel</s>
<|assistant|>
The question asked by the given text material is, "What are some factors that can cause a disc bulge in the spine, and how can these factors be understood to avoid injury risk?" The answer to this question is as follows:

1. Increased weight lifting and poor technique can cause a disc bulge.
2. The mechanism that creates a disc bulge involves ͆exion.
3. The force applied to the spine during 

In [50]:
response = outputs[0]["generated_text"].split('<|assistant|>')[1]
start = response.find("Question:")
end = response.rfind("Answer:") -2
question = response[start:end]
answer = response[response.find("Answer:"):]

In [52]:
print(question)

In [42]:
start = response.find("Question:")
end = response.rfind("Answer:") -2
question = response[start:end]

In [46]:
answer = response[response.find("Answer:"):]
answer

'Answer: "The power generated at the spine increases at one or two speciͅc joints of the lumbar spine (usually L4/5 and L5/S1) during a normal-tempo air squat. This increase in power is a potential risk for injury, as it can lead to stress concentrations at those joints."'

In [ ]:
##### BETTER

In [6]:
def get_prompt(text):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful and concise medical tutor. "
                "Based on the text below, generate a JSON object with a single question-answer pair "
                "using the keys 'instruction' for the question and 'output' for the answer.\n"
                "Respond only with the JSON object."
            ),
        },
        {"role": "user", "content": text},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Function to generate QA from text


In [7]:
prompt = get_prompt(chunks[103].page_content)

output = qa_gen(prompt, max_new_tokens=256, do_sample=True, temperature=0.7, top_k=50, top_p=0.95)[0]["generated_text"]
output.split("<|assistant|>")[1]


'\n{\n  "question": "What is the mechanism behind the development of a disc bulge, and how does it affect injury risk?",\n  "answer": "The mechanism behind the development of a disc bulge is ͆exion, which is the process of ͆exing the spine as it ͆exes. If the force applied to the spine as it ͆exes is low, power generation remains low, and so does injury risk. This is why the cat-camel model, which simulates low-force disc bulging, is a useful tool for understanding the mechanics of disc injury risk."'

'\n{\n  "question": "If you are performing a few bodyweight squats and butt winking occurs, is it a big deal?",\n  "answer": "No, minimal power is generated at the spine during a normal-tempo air squat, but as soon as you add a barbell, things change. If butt winking continues under load, the power generated at the spine increases at one or two specific joints of the lumbar spine (usually L4/5 and L5/S1). Therefore, when you have a stress concentration of power at one or two lumbar segments, injury risk can increase."'

In [ ]:
# --- STEP 4: Generate QA pairs from each chunk
qa_pairs = []

for i, chunk in enumerate(chunks):
    prompt = make_prompt(chunk)
    output = generator(prompt, max_new_tokens=512, do_sample=True, temperature=0.7)[0]['generated_text']

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a friendly chatbot who always responds in the style of a pirate",
    },
    {"role": "user", "content": "How many helicopters can a human eat in one sitting?"},
]
#'<|system|>\nYou are a friendly chatbot who always responds in the style of a pirate</s>\n<|user|>\nHow many helicopters can a human eat in one sitting?</s>\n<|assistant|>\n'
prompt = qa_gen.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)



In [21]:
outputs = qa_gen(prompt, max_new_tokens=256, do_sample=True, temperature=0.7, top_k=50, top_p=0.95)
print(outputs[0]["generated_text"])


<|system|>
You are a friendly chatbot who always responds in the style of a pirate</s>
<|user|>
How many helicopters can a human eat in one sitting?</s>
<|assistant|>
No, there is no direct answer to the question "How many helicopters can a human eat in one sitting?" The answer depends on the size and shape of the helicopter, as well as the type of food it consumes. Helicopters are designed to fly for long periods of time, often for extended periods, without refueling. Therefore, the amount of food they can consume is limited by the amount of fuel they can carry. A small helicopter with a 150-kilogram (330-pound) fuel capacity may be able to consume a few slices of pizza or a sandwich. A larger helicopter with a larger fuel capacity may be able to consume more, but it would also require a larger amount of fuel. It's worth consulting the manufacturer or aviation experts for more detailed information on the specific helicopter you are referring to.


In [26]:
text = chunks[100].page_content

In [27]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant that generates question-answer pairs based on passages",
    },
    {"role": "user", "content": text},
]
#'<|system|>\nYou are a friendly chatbot who always responds in the style of a pirate</s>\n<|user|>\nHow many helicopters can a human eat in one sitting?</s>\n<|assistant|>\n'
prompt = qa_gen.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

outputs = qa_gen(prompt, max_new_tokens=256, do_sample=True, temperature=0.7, top_k=50, top_p=0.95)
print(outputs[0]["generated_text"])


<|system|>
You are a helpful assistant that generates question-answer pairs based on passages</s>
<|user|>
on their spine. This would be labeled “load intolerance.” If this sounds
familiar, not only write down the amount of weight at which you start to
experience symptoms but also analyze the motions and postures you are
assuming during that speciͅc exercise.
Along with your assessment of the gym movements that trigger your
pain, the postures you assume and the movements you put your body
through during the other 22 to 23 hours of the day outside of training are
just as important to evaluate. The accumulated microtrauma that has
caused your current back pain may not be due solely to training.
Think to yourself whether rounded or extended postures create or
alleviate pain in your back. For example, I have many patients who
complain of back pain after sitting all day at work yet have no pain if they
are up and walking (this would point to ͆exion intolerance). However,
some people may hav

In [28]:
outputs[0]["generated_text"].split("<|assistant|>")[-1].strip()


'Sure, here\'s an updated version of the passage with your suggestions added:\n\non their spine. This would be labeled "load intolerance" if you start to experience symptoms at a certain weight or level of motion during certain exercises. During your assessment, analyze the motions and postures you assume during gym movements that trigger your pain, as well as the movements you put your body through outside of training. Think to yourself whether rounded or extended postures create or alleviate pain in your back. For example, I have many patients who complain of back pain after sitting all day at work yet have no pain if they are up and walking (this would point to exion intolerance). However, some people may have pain when walking/running for 15 minutes that is severe.\nAlong with your assessment of the gym movements that trigger your pain, the postures you assume and the movements you put your body through during the other 22 to 23 hours of the day outside of training are just as impo

In [31]:
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForQuestionAnswering.from_pretrained(model_name)
#qa_gen = pipeline("q", model=model, tokenizer=tokenizer,  max_new_tokens=200)


Some weights of LlamaForQuestionAnswering were not initialized from the model checkpoint at TinyLlama/TinyLlama-1.1B-Chat-v1.0 and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight', 'transformer.embed_tokens.weight', 'transformer.layers.0.input_layernorm.weight', 'transformer.layers.0.mlp.down_proj.weight', 'transformer.layers.0.mlp.gate_proj.weight', 'transformer.layers.0.mlp.up_proj.weight', 'transformer.layers.0.post_attention_layernorm.weight', 'transformer.layers.0.self_attn.k_proj.weight', 'transformer.layers.0.self_attn.o_proj.weight', 'transformer.layers.0.self_attn.q_proj.weight', 'transformer.layers.0.self_attn.v_proj.weight', 'transformer.layers.1.input_layernorm.weight', 'transformer.layers.1.mlp.down_proj.weight', 'transformer.layers.1.mlp.gate_proj.weight', 'transformer.layers.1.mlp.up_proj.weight', 'transformer.layers.1.post_attention_layernorm.weight', 'transformer.layers.1.self_attn.k_proj.weight', 'transformer.layers.1.self_attn.o_proj.weight', 'transform

In [32]:
qa_gen = pipeline("document-question-answering", model=model, tokenizer=tokenizer,  max_new_tokens=200)

Device set to use cuda:0
The model 'LlamaForQuestionAnswering' is not supported for document-question-answering. Supported models are ['PeftModelForQuestionAnswering', 'LayoutLMForQuestionAnswering', 'LayoutLMv2ForQuestionAnswering', 'LayoutLMv3ForQuestionAnswering'].


## HuggingFaceH4/zephyr-7b-beta

In [75]:
model_zeph_name = "HuggingFaceH4/zephyr-7b-beta"
tokenizer_z = AutoTokenizer.from_pretrained(model_zeph_name)
model_z = AutoModelForCausalLM.from_pretrained(model_zeph_name)


Loading checkpoint shards:  62%|██████▎   | 5/8 [00:11<00:07,  2.43s/it]

: 

In [ ]:
pipe = pipeline("text-generation", model="HuggingFaceH4/zephyr-7b-beta", torch_dtype=torch.bfloat16, device_map="auto")

In [7]:
text = chunks[10].page_content

In [8]:

# We use the tokenizer's chat template to format each message - see https://huggingface.co/docs/transformers/main/en/chat_templating
messages = [
    {
        "role": "system",
        "content": "You are an assistant that generates high-quality question-answer pairs for fine-tuning. Only generate output related to health, fitness, anatomy, injuries, or sports",
    },
    {"role": "user", "content": f"{text}"},
]

In [15]:
model_id = "HuggingFaceH4/zephyr-7b-beta"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Define generation pipeline
qa_gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
    top_p=0.95,
    repetition_penalty=1.1
)


Loading checkpoint shards: 100%|██████████| 8/8 [00:05<00:00,  1.41it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.
Device set to use cuda:0


In [16]:
text

'Introduction\nBehind many a legendary tale, there is a\nhidden lesson to be learned. Those who\nincorporate weight training with the goal\nof improving their physique, strength, or\npower should look to the story of Milo of\nCroton.\nMilo was an ancient Greek Olympian and the poster child for athletic\nexcellence in his time. Legend has it that at a young age, Milo started his\nstrength journey by lifting a small calf and carrying it on his shoulders\nevery day. As the calf grew over the years, so did Milo’s strength, until one\nday he was hoisting a full-grown bull! Although it is unlikely that he was\nable to lift a 1,500-pound bull, like any good story, the legend sprang\nfrom humble beginnings.\nThe 2,500-year-old story of how Milo built his heroic strength set forth\nthe training principle that all athletes adhere to. That principle is now\nknown as “progressive overload.” Within Milo’s story is the idea that hard\nwork combined with consistency can lead to legendary feats of str

In [17]:
prompt = f"""<|system|>
You are a helpful assistant that generates high-quality question-answer pairs for fine-tuning.
Only generate output related to health, fitness, anatomy, injuries, or sports. Format the result as JSON: {{"instruction": "<question>", "output": "<answer>"}}.
If irrelevant, return: {{"instruction": null, "output": null}}.
<|user|>
Generate one instruction and answer from the following passage:
\"\"\"{text}\"\"\"
<|assistant|>
"""

# Generate output
response = qa_gen(prompt)[0]["generated_text"]

OutOfMemoryError: CUDA out of memory. Tried to allocate 250.00 MiB. GPU 0 has a total capacity of 11.64 GiB of which 211.81 MiB is free. Including non-PyTorch memory, this process has 11.43 GiB memory in use. Of the allocated memory 11.24 GiB is allocated by PyTorch, and 53.31 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch
import json

# Load Zephyr model and tokenizer
model_id = "HuggingFaceH4/zephyr-7b-beta"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Define generation pipeline
qa_gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
    top_p=0.95,
    repetition_penalty=1.1
)

# Sample input chunk
chunk_text = """
Resistance training involves using resistance to muscular contraction to build strength, anaerobic endurance, and muscle size. It includes exercises like weightlifting and bodyweight routines.
"""

# Chat-style prompt for Zephyr
prompt = f"""<|system|>
You are a helpful assistant that generates high-quality question-answer pairs for fine-tuning.
Only generate output related to health, fitness, anatomy, injuries, or sports. Format the result as JSON: {{"instruction": "<question>", "output": "<answer>"}}.
If irrelevant, return: {{"instruction": null, "output": null}}.
<|user|>
Generate one instruction and answer from the following passage:
\"\"\"{chunk_text}\"\"\"
<|assistant|>
"""

# Generate output
response = qa_gen(prompt)[0]["generated_text"]

# Extract only the assistant's JSON
generated_part = response.split("<|assistant|>")[-1].strip()

# Try to parse JSON
try:
    result = json.loads(generated_part)
except json.JSONDecodeError:
    print("⚠️ Failed to parse model output.")
    result = {"instruction": None, "output": None}

# Display result
print(json.dumps(result, indent=2))


Loading checkpoint shards: 100%|██████████| 8/8 [00:04<00:00,  1.63it/s]
Some parameters are on the meta device because they were offloaded to the cpu.
Device set to use cuda:0


⚠️ Failed to parse model output.
{
  "instruction": null,
  "output": null
}


## Generic Example of QA generation from text

In [18]:
input_text = """
Resistance training, also known as strength training, involves the performance of physical exercises that are designed to improve strength and endurance. It is a key component in bodybuilding and powerlifting.
"""

prompt = f"""Based on the following context, generate a list of questions and their answers:\n\nContext:\n{input_text}\n\nQ1:"""

output = qa_gen(prompt, do_sample=True, temperature=0.7)[0]['generated_text']

print(output)


Based on the following context, generate a list of questions and their answers:

Context:

Resistance training, also known as strength training, involves the performance of physical exercises that are designed to improve strength and endurance. It is a key component in bodybuilding and powerlifting.


Q1: What is resistance training and how does it relate to bodybuilding and powerlifting?
A1: Resistance training refers to the use of weights, resistance bands, or machines to improve strength and endurance. It is a key component in bodybuilding and powerlifting.

Q2: What are some popular types of resistance training?
A2: Some popular types of resistance training include:

- Dumbbell exercises (such as bench press, bicep curls, and squats)
- Barbell exercises (such as deadlifts, squat jumps, and bench presses)
- Resistance bands (such as pull-ups, bicep curls, and leg presses)
- Machines (such as leg press, shoulder press, and tricep extension)

Q3: Which exercises are best for beginners

In [19]:
output

'Based on the following context, generate a list of questions and their answers:\n\nContext:\n\nResistance training, also known as strength training, involves the performance of physical exercises that are designed to improve strength and endurance. It is a key component in bodybuilding and powerlifting.\n\n\nQ1: What is resistance training and how does it relate to bodybuilding and powerlifting?\nA1: Resistance training refers to the use of weights, resistance bands, or machines to improve strength and endurance. It is a key component in bodybuilding and powerlifting.\n\nQ2: What are some popular types of resistance training?\nA2: Some popular types of resistance training include:\n\n- Dumbbell exercises (such as bench press, bicep curls, and squats)\n- Barbell exercises (such as deadlifts, squat jumps, and bench presses)\n- Resistance bands (such as pull-ups, bicep curls, and leg presses)\n- Machines (such as leg press, shoulder press, and tricep extension)\n\nQ3: Which exercises are

In [14]:
qa_gen("Hello, what are you?", max_new_tokens=25)

[{'generated_text': 'Hello, what are you?\n\nJESSICA: (smiling) Good question.\n\nMARC: (looking at her)'}]

In [ ]:
qa_gen(["Hello, what are you?", "Where is Italy?"], max_new_tokens=25) 

[[{'generated_text': 'Hello, what are you?\n\nJASON: I’m a firefighter.\n\nCHRISTINA: That’'}],
 [{'generated_text': 'Where is Italy?'}]]

## Create QA specific format for LLama

In [20]:
text = chunks[10].page_content

In [21]:
def get_promp(text):
    messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant generating study questions and answers",
    },
    {"role": "user", "content": text},
]
    return messages
prompt = qa_gen.tokenizer.apply_chat_template(get_promp(text), tokenize=False, add_generation_prompt=True)


In [23]:
outputs = qa_gen(prompt, max_new_tokens=256, do_sample=True, temperature=0.7, top_k=50, top_p=0.95)
print(outputs[0]["generated_text"])


<|system|>
You are a helpful assistant generating study questions and answers</s>
<|user|>
Introduction
Behind many a legendary tale, there is a
hidden lesson to be learned. Those who
incorporate weight training with the goal
of improving their physique, strength, or
power should look to the story of Milo of
Croton.
Milo was an ancient Greek Olympian and the poster child for athletic
excellence in his time. Legend has it that at a young age, Milo started his
strength journey by lifting a small calf and carrying it on his shoulders
every day. As the calf grew over the years, so did Milo’s strength, until one
day he was hoisting a full-grown bull! Although it is unlikely that he was
able to lift a 1,500-pound bull, like any good story, the legend sprang
from humble beginnings.
The 2,500-year-old story of how Milo built his heroic strength set forth
the training principle that all athletes adhere to. That principle is now
known as “progressive overload.” Within Milo’s story is the idea th

In [24]:
outputs[0]["generated_text"].split('<|assistant|>')[1]

'\nIntroduction:\n\nBehind many a legendary tale, there is a hidden lesson to be learned. Those who incorporate weight training with the goal of improving their physique, strength, or power should look to the story of Milo of Croton. Milo was an ancient Greek Olympian and the poster child for athletic excellence in his time. Legend has it that at a young age, Milo started his strength journey by lifting a small calf and carrying it on his shoulders every day. As the calf grew over the years, so did Milo\'s strength, until one day he was hoisting a full-grown bull! Although it is unlikely that he was able to lift a 1,500-pound bull, like any good story, the legend sprang from humble beginnings.\n\nThe 2,500-year-old story of how Milo built his heroic strength set forth the training principle that all athletes adhere to. That principle is now known as "progressive overload." Within Milo\'s story, he demonstrated the idea that hard work combined with consistency can lead to legendary feat

In [ ]:
## Create QA specific format for LLama
def get_promp(text):
    # prompt = f"""
    # Given the following passage, create a question a student might ask, and answer it.
    # Return the result as a JSON object with 'question' labeled as 'instruction' and 'answer', labeled as 'output'.
    # If the text should only be related to health, fitness, phisioterapy, body anatomy, ingiuries, and sport related topics.
    # Text: '{text.page_content}'
    # """
    prompt = f"""
        You are a helpful assistant generating study questions and answers.

        Given the following passage, create a realistic question that a student might ask, and provide a concise, informative answer.

        Only generate a question if the content is clearly related to one or more of the following topics:
        - Health
        - Fitness
        - Physiotherapy
        - Body anatomy
        - Injuries
        - Sports or athletic performance

        If the passage is off-topic, respond with: {{ "instruction": null, "output": null }}

        Return the result as a JSON object with:
        - "instruction" = the question
        - "input" = ""
        - "output" = the answer

        Text:
        \"\"\"{text}\"\"\"
        """

    return prompt
def get_promp(text):
    prompt = f"""
    You are a helpful assistant generating study questions and answers.

    Given the following passage, create a realistic question a student might ask, and provide a concise, informative answer.

    Only respond if the passage is clearly related to:
    - Health
    - Fitness
    - Physiotherapy
    - Body anatomy
    - Injuries
    - Sports or athletic performance

    Return ONLY the result as a valid JSON object in the following format:
    {{ "instruction": "<question>", "output": "<answer>" }}

    If the passage is not relevant, return:
    {{ "instruction": null, "output": null }}

    Passage:{text}
    """



    return prompt
    input=chunks[10].page_content
    output = qa_gen(get_promp(input), do_sample=True, temperature=0.7)[0]['generated_text']

    print(output)

response.split("<|assistant|>")[-1].strip()

response = output
try:
    start = response.index('{')
    end = response.rindex('}') + 1
    qa_pair = json.loads(response[start:end])
except Exception as e:
    qa_pair = {"instruction": None, "output": None}
    print("Error parsing JSON:", e)

# Example result
print(qa_pair) 
def format_chat_prompt(message: str) -> str:
    return f"<|system|>\nYou are a helpful assistant that generates question-answer pairs based on passages.\n" \
           f"<|user|>\n{message}\n<|assistant|>\n"

def get_promp(passage):
    prompt = f"""Generate a JSON with a study question and a concise answer based on this passage. 
    Only respond if the content is about health, fitness, anatomy, or sports.
    Return the result as a JSON object with 'question' labeled as 'instruction' and 'answer', labeled as 'output'
    Passage: {passage}"""
    return prompt

text=chunk[10].page_content
response = qa_gen(get_promp(text))[0]["generated_text"]
response

## Create QA specific format for LLama

In [ ]:
def get_promp(text):
    # prompt = f"""
    # Given the following passage, create a question a student might ask, and answer it.
    # Return the result as a JSON object with 'question' labeled as 'instruction' and 'answer', labeled as 'output'.
    # If the text should only be related to health, fitness, phisioterapy, body anatomy, ingiuries, and sport related topics.
    # Text: '{text.page_content}'
    # """
    prompt = f"""
        You are a helpful assistant generating study questions and answers.

        Given the following passage, create a realistic question that a student might ask, and provide a concise, informative answer.

        Only generate a question if the content is clearly related to one or more of the following topics:
        - Health
        - Fitness
        - Physiotherapy
        - Body anatomy
        - Injuries
        - Sports or athletic performance

        If the passage is off-topic, respond with: {{ "instruction": null, "output": null }}

        Return the result as a JSON object with:
        - "instruction" = the question
        - "input" = ""
        - "output" = the answer

        Text:
        \"\"\"{text.page_content}\"\"\"
        """

    return prompt

In [ ]:
# from langchain.text_splitter import SemanticChunker
# from langchain.embeddings import OpenAIEmbeddings

# embedding = OpenAIEmbeddings()
# splitter = SemanticChunker(embedding)

# chunks = splitter.split_documents(document)


In [ ]:
chunk[10].page_content

'Introduction\nBehind many a legendary tale, there is a\nhidden lesson to be learned. Those who\nincorporate weight training with the goal\nof improving their physique, strength, or\npower should look to the story of Milo of\nCroton.\nMilo was an ancient Greek Olympian and the poster child for athletic\nexcellence in his time. Legend has it that at a young age, Milo started his\nstrength journey by lifting a small calf and carrying it on his shoulders\nevery day. As the calf grew over the years, so did Milo’s strength, until one\nday he was hoisting a full-grown bull! Although it is unlikely that he was\nable to lift a 1,500-pound bull, like any good story, the legend sprang\nfrom humble beginnings.\nThe 2,500-year-old story of how Milo built his heroic strength set forth\nthe training principle that all athletes adhere to. That principle is now\nknown as “progressive overload.” Within Milo’s story is the idea that hard\nwork combined with consistency can lead to legendary feats of str

In [ ]:
# input_text = """
# Resistance training, also known as strength training, involves the performance of physical exercises that are designed to improve strength and endurance. It is a key component in bodybuilding and powerlifting.
# """

prompt = f"""Based on the following context, generate a list of questions and their answers:\n\nContext:\n{chunk[10].page_content}\n\nQ1:"""

output = qa_gen(prompt, do_sample=True, temperature=0.7)[0]['generated_text']

print(output)


Based on the following context, generate a list of questions and their answers:

Context:
Introduction
Behind many a legendary tale, there is a
hidden lesson to be learned. Those who
incorporate weight training with the goal
of improving their physique, strength, or
power should look to the story of Milo of
Croton.
Milo was an ancient Greek Olympian and the poster child for athletic
excellence in his time. Legend has it that at a young age, Milo started his
strength journey by lifting a small calf and carrying it on his shoulders
every day. As the calf grew over the years, so did Milo’s strength, until one
day he was hoisting a full-grown bull! Although it is unlikely that he was
able to lift a 1,500-pound bull, like any good story, the legend sprang
from humble beginnings.
The 2,500-year-old story of how Milo built his heroic strength set forth
the training principle that all athletes adhere to. That principle is now
known as “progressive overload.” Within Milo’s story is the idea tha

In [ ]:
def get_promp(text):
    prompt = f"""
    You are a helpful assistant generating study questions and answers.

    Given the following passage, create a realistic question a student might ask, and provide a concise, informative answer.

    Only respond if the passage is clearly related to:
    - Health
    - Fitness
    - Physiotherapy
    - Body anatomy
    - Injuries
    - Sports or athletic performance

    Return ONLY the result as a valid JSON object in the following format:
    {{ "instruction": "<question>", "output": "<answer>" }}

    If the passage is not relevant, return:
    {{ "instruction": null, "output": null }}

    Passage:{text}
    """



    return prompt

In [ ]:
input=chunks[10].page_content
output = qa_gen(get_promp(input), do_sample=True, temperature=0.7)[0]['generated_text']

print(output)



    You are a helpful assistant generating study questions and answers.

    Given the following passage, create a realistic question a student might ask, and provide a concise, informative answer.

    Only respond if the passage is clearly related to:
    - Health
    - Fitness
    - Physiotherapy
    - Body anatomy
    - Injuries
    - Sports or athletic performance

    Return ONLY the result as a valid JSON object in the following format:
    { "instruction": "<question>", "output": "<answer>" }

    If the passage is not relevant, return:
    { "instruction": null, "output": null }

    Passage:Introduction
Behind many a legendary tale, there is a
hidden lesson to be learned. Those who
incorporate weight training with the goal
of improving their physique, strength, or
power should look to the story of Milo of
Croton.
Milo was an ancient Greek Olympian and the poster child for athletic
excellence in his time. Legend has it that at a young age, Milo started his
strength journey by

In [ ]:
response.split("<|assistant|>")[-1].strip()


'You are a helpful assistant generating study questions and answers.\n\n    Given the following passage, create a realistic question a student might ask, and provide a concise, informative answer.\n\n    Only respond if the passage is clearly related to:\n    - Health\n    - Fitness\n    - Physiotherapy\n    - Body anatomy\n    - Injuries\n    - Sports or athletic performance\n\n    Return ONLY the result as a valid JSON object in the following format:\n    { "instruction": "<question>", "output": "<answer>" }\n\n    If the passage is not relevant, return:\n    { "instruction": null, "output": null }\n\n    Passage:\n    """Introduction\nBehind many a legendary tale, there is a\nhidden lesson to be learned. Those who\nincorporate weight training with the goal\nof improving their physique, strength, or\npower should look to the story of Milo of\nCroton.\nMilo was an ancient Greek Olympian and the poster child for athletic\nexcellence in his time. Legend has it that at a young age, Milo 

In [47]:
response = output
try:
    start = response.index('{')
    end = response.rindex('}') + 1
    qa_pair = json.loads(response[start:end])
except Exception as e:
    qa_pair = {"instruction": None, "output": None}
    print("Error parsing JSON:", e)

# Example result
print(qa_pair) 

Error parsing JSON: Extra data: line 3 column 5 (char 59)
{'instruction': None, 'output': None}


In [56]:
def format_chat_prompt(message: str) -> str:
    return f"<|system|>\nYou are a helpful assistant that generates question-answer pairs based on passages.\n" \
           f"<|user|>\n{message}\n<|assistant|>\n"


In [63]:
def get_promp(passage):
    prompt = f"""Generate a JSON with a study question and a concise answer based on this passage. 
    Only respond if the content is about health, fitness, anatomy, or sports.
    Return the result as a JSON object with 'question' labeled as 'instruction' and 'answer', labeled as 'output'
    Passage: {passage}"""
    return prompt


In [64]:
text=chunk[10].page_content
response = qa_gen(get_promp(text))[0]["generated_text"]
response

"Generate a JSON with a study question and a concise answer based on this passage. \n    Only respond if the content is about health, fitness, anatomy, or sports.\n    Return the result as a JSON object with 'question' labeled as 'instruction' and 'answer', labeled as 'output'\n    Passage: Introduction\nBehind many a legendary tale, there is a\nhidden lesson to be learned. Those who\nincorporate weight training with the goal\nof improving their physique, strength, or\npower should look to the story of Milo of\nCroton.\nMilo was an ancient Greek Olympian and the poster child for athletic\nexcellence in his time. Legend has it that at a young age, Milo started his\nstrength journey by lifting a small calf and carrying it on his shoulders\nevery day. As the calf grew over the years, so did Milo’s strength, until one\nday he was hoisting a full-grown bull! Although it is unlikely that he was\nable to lift a 1,500-pound bull, like any good story, the legend sprang\nfrom humble beginnings.\

In [60]:
output = response.split("<|assistant|>")[-1].strip()
output

'{\n    "instruction": "Generate a study question based on the text material about Milo of Croton\'s weight training journey. The question should be about health, fitness, anatomy, or sports. Return only if the content is relevant to these topics and in English.",\n    "output": "What is the principle that Milo of Croton\'s weight training journey followed, and how does it relate to strength development?"\n}'

In [73]:
import re
import json

def extract_qa_from_chunk(chunk_text, qa_gen):
    prompt = f"""
Generate a relevant question and a concise answer based on the text below, in JSON format:
{{ "instruction": "<question>", "output": "<answer>" }}

Only respond with the JSON. If the content is not related to health, fitness, or anatomy, return:
{{ "instruction": null, "output": null }}

Text:
\"\"\"{chunk_text}\"\"\"
"""
    response = qa_gen(prompt, do_sample=True, temperature=0.7, max_new_tokens=200)[0]['generated_text']


    return response

# Example usage over many chunks
all_chunks = [chunks[10].page_content, chunks[11].page_content]  # list of text.page_content or strings
qa_pairs = []

for c in all_chunks:
    result = extract_qa_from_chunk(c, qa_gen)
    #if result["instruction"] and result["output"]:
    qa_pairs.append(result)

# Output: list of dicts with instruction/output
print(qa_pairs)


['\nGenerate a relevant question and a concise answer based on the text below, in JSON format:\n{ "instruction": "<question>", "output": "<answer>" }\n\nOnly respond with the JSON. If the content is not related to health, fitness, or anatomy, return:\n{ "instruction": null, "output": null }\n\nText:\n"""Introduction\nBehind many a legendary tale, there is a\nhidden lesson to be learned. Those who\nincorporate weight training with the goal\nof improving their physique, strength, or\npower should look to the story of Milo of\nCroton.\nMilo was an ancient Greek Olympian and the poster child for athletic\nexcellence in his time. Legend has it that at a young age, Milo started his\nstrength journey by lifting a small calf and carrying it on his shoulders\nevery day. As the calf grew over the years, so did Milo’s strength, until one\nday he was hoisting a full-grown bull! Although it is unlikely that he was\nable to lift a 1,500-pound bull, like any good story, the legend sprang\nfrom humble

In [68]:
result

{'instruction': None, 'output': None}

In [74]:
input_text = """
Resistance training, also known as strength training, involves the performance of physical exercises that are designed to improve strength and endurance. It is a key component in bodybuilding and powerlifting.
"""

prompt = f"""Based on the following context, generate a list of questions and their answers:\n\nContext:\n{chunks[10].page_content}\n\nQ1:"""

output = qa_gen(prompt, do_sample=True, temperature=0.7)[0]['generated_text']

print(output)


Based on the following context, generate a list of questions and their answers:

Context:
Introduction
Behind many a legendary tale, there is a
hidden lesson to be learned. Those who
incorporate weight training with the goal
of improving their physique, strength, or
power should look to the story of Milo of
Croton.
Milo was an ancient Greek Olympian and the poster child for athletic
excellence in his time. Legend has it that at a young age, Milo started his
strength journey by lifting a small calf and carrying it on his shoulders
every day. As the calf grew over the years, so did Milo’s strength, until one
day he was hoisting a full-grown bull! Although it is unlikely that he was
able to lift a 1,500-pound bull, like any good story, the legend sprang
from humble beginnings.
The 2,500-year-old story of how Milo built his heroic strength set forth
the training principle that all athletes adhere to. That principle is now
known as “progressive overload.” Within Milo’s story is the idea tha